# 31 — Chỉ báo kỹ thuật và bốn cái bẫy

Mở đầu Track 3. finlens có **135 hàm TA-Lib** ở hai tầng, và chọn nhầm tầng là
một trong những cách hỏng đắt nhất trong cả thư viện — vì nó hỏng *im lặng*,
ra những con số nằm trọn trong khoảng hợp lệ.

| Tầng | Gọi thế nào | Khi nào dùng |
|---|---|---|
| `df.finlens.*` | `df.finlens.rsi(14)` | **mặc định** — tự tách theo mã, tự cảnh báo |
| `finlens.ta.*` | `finlens.ta.RSI(close, timeperiod=14)` | một mảng, một chuỗi giá duy nhất |

Notebook này đo bốn cái bẫy bằng số thật:

1. **Nhiều mã trong một frame** — sai ở phần lớn số dòng của mã thứ hai
2. **Warm-up** — chỉ báo ra `NaN`, mẫu nến ra `0`, hai thứ khác nhau
3. **`NaN` ở giữa lan tới hết chuỗi**
4. **24 trên 75 chỉ báo có trạng thái không ổn định**

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Tầng `df.finlens.*` — nối chuỗi được, tự tách nhóm

Mọi method trả về một **bản sao** kèm cột mới, nên nối chuỗi được. Tên cột mang
theo tham số, nên `sma(20)` và `sma(50)` là hai cột chứ không đè lên nhau.

In [2]:
gia = client.eod.stock.ohlcv(["HPG", "VCB"], start=lui_ngay(HOM_NAY, nam=1))

co_chi_bao = gia.finlens.rsi(14).finlens.macd().finlens.bbands(20).finlens.sma(20).finlens.sma(50)

print(f"{gia.shape[1]} cột vào → {co_chi_bao.shape[1]} cột ra")
print([c for c in co_chi_bao.columns if c not in gia.columns])

7 cột vào → 16 cột ra
['rsi_14', 'macd_12_26_9', 'macdsignal_12_26_9', 'macdhist_12_26_9', 'upperband_20_2_2_0', 'middleband_20_2_2_0', 'lowerband_20_2_2_0', 'sma_20', 'sma_50']


`df.finlens` được đăng ký khi `finlens.accessor` được nạp, và `finlens.client()`
nạp nó. Nếu bạn dựng `DataFrame` từ file parquet mà không tạo client thì cần
`import finlens.accessor` một lần.

## 2 · ⚠️ Bẫy 1 — chỉ báo trên frame nhiều mã

Đây là bẫy tốn kém nhất. Frame ở dạng long, hai mã xếp chồng lên nhau. Gọi
TA-Lib thẳng trên cột `close` thì cửa sổ 14 phiên đầu của mã thứ hai **ăn 13
giá cuối của mã thứ nhất**.

In [3]:
import talib  # gọi thẳng TA-Lib để dựng lại cách sai

hai_ma = gia.sort_values(["symbol", "date"]).reset_index(drop=True)

cach_sai = talib.RSI(hai_ma["close"].to_numpy(), timeperiod=14)
cach_dung = hai_ma.finlens.rsi(14)["rsi_14"].to_numpy()

khac = ~np.isclose(cach_sai, cach_dung, equal_nan=True)

print(f"Frame: {len(hai_ma)} dòng, {hai_ma['symbol'].nunique()} mã")
print(f"Số dòng khác nhau: {khac.sum()} / {len(hai_ma)}  ({khac.mean():.0%})\n")

theo_ma = hai_ma.assign(khac=khac).groupby("symbol", observed=True)["khac"].agg(["sum", "count"])
for ma_ck, r in theo_ma.iterrows():
    print(f"  {ma_ck}: {int(r['sum']):>4} / {int(r['count'])} dòng sai")

print(f"\nLệch lớn nhất: {np.nanmax(np.abs(cach_sai - cach_dung)):.2f} điểm RSI")
print(f"Khoảng giá trị của các số SAI: {np.nanmin(cach_sai[khac]):.1f} … {np.nanmax(cach_sai[khac]):.1f}")
print("→ nằm trọn trong 0–100. Không có gì để một phép kiểm tự động bắt được.")

Frame: 500 dòng, 2 mã
Số dòng khác nhau: 174 / 500  (35%)

  HPG:    0 / 250 dòng sai
  VCB:  174 / 250 dòng sai

Lệch lớn nhất: 18.11 điểm RSI
Khoảng giá trị của các số SAI: 29.2 … 96.2
→ nằm trọn trong 0–100. Không có gì để một phép kiểm tự động bắt được.


In [4]:
doi_chieu = hai_ma.assign(sai=cach_sai, dung=cach_dung)
mo_dau_ma_2 = doi_chieu[doi_chieu["symbol"] == hai_ma["symbol"].unique()[1]].head(8)
mo_dau_ma_2[["symbol", "date", "close", "sai", "dung"]].assign(
    sai=lambda d: d["sai"].round(2), dung=lambda d: d["dung"].round(2)
)

,symbol,date,close,sai,dung
250,VCB,2025-08-11,61.53,96.16,NaN
251,VCB,2025-08-12,62.03,96.21,NaN
252,VCB,2025-08-13,61.53,94.93,NaN
253,VCB,2025-08-14,64.19,95.29,NaN
254,VCB,2025-08-15,63.31,92.96,NaN
255,VCB,2025-08-18,63.50,93.00,NaN
256,VCB,2025-08-19,63.21,92.14,NaN
257,VCB,2025-08-20,62.03,88.57,NaN


Đọc kỹ tám dòng trên. Ở tám phiên đầu của mã thứ hai, cách sai cho ra một con
số RSI trông hoàn toàn bình thường, còn cách đúng cho ra `NaN` — vì mười bốn
phiên đầu tiên **chưa đủ dữ liệu để tính**.

Con số đó không phải của mã này. Nó là dư âm giá của mã đứng trước nó trong
frame, đi qua công thức RSI.

⚠️ Chú ý dòng in ở trên: **mã đầu tiên đúng hoàn toàn, mã thứ hai sai gần hết**.
Vùng sai không dừng ở 14 phiên warm-up — RSI làm trơn theo cấp số nhân nên
nhiễm bẩn tắt dần chứ không tắt hẳn, và độ dài vùng sai phụ thuộc chuỗi. Đó
chính là bẫy số 4 ở phần dưới đang tác động cùng lúc.

In [5]:
ve = doi_chieu[doi_chieu["symbol"] == hai_ma["symbol"].unique()[1]].head(60)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ve["date"], y=ve["sai"], name="talib.RSI trên cả cột close (SAI)",
                         line=dict(width=2, color=GIAM)))
fig.add_trace(go.Scatter(x=ve["date"], y=ve["dung"], name="df.finlens.rsi(14) (ĐÚNG)",
                         line=dict(width=2, color=CHUOI[0])))
fig.add_hline(y=70, line_dash="dot", line_width=1, line_color="#898781")
fig.add_hline(y=30, line_dash="dot", line_width=1, line_color="#898781")
fig.update_layout(
    title_text=f"RSI 60 phiên đầu của {hai_ma['symbol'].unique()[1]} trong frame hai mã<br>"
    "<sub style='color:#52514e'>Đường đỏ bắt đầu ở vùng quá mua vì nó thừa hưởng giá của mã đứng trước</sub>",
    yaxis_title="RSI",
    height=460,
    hovermode="x unified",
)
fig

### Vì sao tầng `df.finlens.*` tồn tại

Nó tự dò cột khoá (`symbol`, `icb`, `code`) rồi tính riêng từng nhóm. Nếu frame
của bạn thật sự là **một** chuỗi giá duy nhất thì truyền `by=None`.

In [6]:
mot_ma = gia[gia["symbol"] == "HPG"].sort_values("date")

a = mot_ma.finlens.rsi(14)["rsi_14"].to_numpy()
b = mot_ma.finlens.rsi(14, by=None)["rsi_14"].to_numpy()
c = finlens.ta.RSI(mot_ma["close"], timeperiod=14).to_numpy()

print(f"Một mã — ba cách gọi cho cùng kết quả: {np.allclose(a, b, equal_nan=True) and np.allclose(b, c, equal_nan=True)}")

Một mã — ba cách gọi cho cùng kết quả: True


## 3 · ⚠️ Bẫy 2 — warm-up, và hai cách biểu diễn khác nhau

Chỉ báo cần một số thanh để "khởi động". Chúng biểu diễn điều đó theo hai cách
**không tương thích nhau**:

In [7]:
mot = gia[gia["symbol"] == "HPG"].sort_values("date").reset_index(drop=True)

warm = pd.DataFrame(
    {
        "rsi_14": mot.finlens.rsi(14)["rsi_14"],
        "sma_50": mot.finlens.sma(50)["sma_50"],
        "cdldoji": mot.finlens.pattern.cdldoji()["cdldoji"],
    }
)

print("Số ô NaN ở đầu chuỗi:")
print(f"  rsi_14  : {warm['rsi_14'].isna().sum():>3}  (timeperiod=14)")
print(f"  sma_50  : {warm['sma_50'].isna().sum():>3}  (timeperiod=50)")
print(f"  cdldoji : {warm['cdldoji'].isna().sum():>3}  ← mẫu nến KHÔNG dùng NaN")
print(f"\ncdldoji 12 giá trị đầu: {warm['cdldoji'].head(12).tolist()}")

Số ô NaN ở đầu chuỗi:
  rsi_14  :  14  (timeperiod=14)
  sma_50  :  49  (timeperiod=50)
  cdldoji :   0  ← mẫu nến KHÔNG dùng NaN

cdldoji 12 giá trị đầu: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


**Chỉ báo ra `NaN`** — đọc được là "chưa tính được".

**Mẫu nến ra số `0`** — và `0` ở đây không phân biệt được với "đã quét xong,
không có mẫu nào". Cả vùng warm-up lẫn vùng không có mẫu đều là `0`.

Hệ quả: `(df["cdldoji"] == 0).mean()` không phải là "tỷ lệ phiên không có mẫu
doji" — nó gộp cả những phiên chưa quét được.

### Nhóm ngắn hơn warm-up: toàn `NaN`, kèm một cảnh báo

Đây là chỗ tầng `df.finlens.*` làm hộ bạn việc TA-Lib không làm.

In [8]:
import warnings

ngan = gia[gia["symbol"] == "HPG"].sort_values("date").tail(20)

with warnings.catch_warnings(record=True) as bat:
    warnings.simplefilter("always")
    ket_qua = ngan.finlens.sma(50)

if bat:
    print(f"{bat[0].category.__name__}: {bat[0].message}")
print(f"\nSố giá trị không NaN: {ket_qua['sma_50'].notna().sum()} / {len(ket_qua)}")
print(f"TA-Lib gọi thẳng thì im lặng: toàn NaN = {np.isnan(talib.SMA(ngan['close'].to_numpy(), 50)).all()}")


Số giá trị không NaN: 0 / 20
TA-Lib gọi thẳng thì im lặng: toàn NaN = True


## 4 · ⚠️ Bẫy 3 — một `NaN` ở giữa làm hỏng phần còn lại

Không phải một ô hỏng. **Toàn bộ phần sau nó** hỏng, vĩnh viễn.

In [9]:
thu = mot["close"].copy()
VI_TRI = 100
thu.iloc[VI_TRI] = np.nan

ket = finlens.ta.SMA(thu, timeperiod=20)

print(f"Chuỗi {len(thu)} phần tử, đặt NaN ở vị trí {VI_TRI}")
print(f"  số NaN trong đầu vào : {thu.isna().sum()}")
print(f"  số NaN trong đầu ra  : {ket.isna().sum()}")
print(f"  giá trị không NaN cuối cùng ở vị trí: {ket.notna()[::-1].idxmax() if ket.notna().any() else 'không có'}")
print(f"\n→ từ vị trí {VI_TRI} tới hết chuỗi ({len(thu) - VI_TRI} phần tử) đều là NaN")

Chuỗi 250 phần tử, đặt NaN ở vị trí 100
  số NaN trong đầu vào : 1
  số NaN trong đầu ra  : 169
  giá trị không NaN cuối cùng ở vị trí: 99

→ từ vị trí 100 tới hết chuỗi (150 phần tử) đều là NaN


Thư viện **cảnh báo chứ không tự chữa**, và đó là lựa chọn đúng:

- `ffill` là **bịa số** — bạn tạo ra một phiên giao dịch chưa từng tồn tại
- `dropna` là **đổi cửa sổ** — SMA 20 của bạn thành SMA của 20 phiên *có dữ
  liệu*, không phải 20 phiên liên tiếp

Cả hai đều là quyết định của người phân tích, không phải của thư viện.

⚠️ Trong thực tế `NaN` ở giữa đến từ đâu? Thường là từ chính bạn: `merge` với
một frame thiếu vài phiên, `reindex` theo lịch dương thay vì lịch giao dịch,
hoặc ghép hai nguồn có ngày nghỉ khác nhau.

## 5 · ⚠️ Bẫy 4 — 24 trên 75 chỉ báo có trạng thái không ổn định

"Không ổn định" nghĩa là: **cắt bớt phần đầu chuỗi làm đổi kết quả ở phần còn
lại**, không chỉ ở vùng warm-up. Đổi `start=` của lời gọi dữ liệu là đổi con số
bạn nhận về.

In [10]:
ten_ham = [n for n in dir(finlens.ta) if n.isupper()]
khong_on_dinh = [n for n in ten_ham if "trạng thái không ổn định" in (getattr(finlens.ta, n).__doc__ or "")]

print(f"{len(khong_on_dinh)} / {len(ten_ham)} hàm mang cảnh báo này trong docstring:")
print(", ".join(sorted(khong_on_dinh)))

24 / 75 hàm mang cảnh báo này trong docstring:
ADX, ADXR, ATR, CMO, DX, EMA, HT_DCPERIOD, HT_DCPHASE, HT_PHASOR, HT_SINE, HT_TRENDLINE, HT_TRENDMODE, IMI, KAMA, MAMA, MFI, MINUS_DI, MINUS_DM, NATR, PLUS_DI, PLUS_DM, RSI, STOCHRSI, T3


Đo trực tiếp: tính chỉ báo trên **cả chuỗi** rồi cắt 200 phần tử đầu, so với
tính trên **chuỗi đã cắt**. Nếu hàm ổn định, hai kết quả phải trùng khớp.

In [11]:
dai = client.eod.stock.ohlcv("HPG", start=lui_ngay(HOM_NAY, nam=3)).sort_values("date")
c = dai["close"].reset_index(drop=True)
CAT = 200

so_sanh = []
for ten, tham_so in [("SMA", {"timeperiod": 50}), ("EMA", {"timeperiod": 50}),
                     ("RSI", {"timeperiod": 14}), ("ADX", {"timeperiod": 14})]:
    ham = getattr(finlens.ta, ten)
    if ten == "ADX":
        h, l = dai["high"].reset_index(drop=True), dai["low"].reset_index(drop=True)
        toan_bo = ham(h, l, c, **tham_so).to_numpy()[CAT:]
        da_cat = ham(h[CAT:].reset_index(drop=True), l[CAT:].reset_index(drop=True),
                     c[CAT:].reset_index(drop=True), **tham_so).to_numpy()
    else:
        toan_bo = ham(c, **tham_so).to_numpy()[CAT:]
        da_cat = ham(c[CAT:].reset_index(drop=True), **tham_so).to_numpy()

    hop_le = ~np.isnan(toan_bo) & ~np.isnan(da_cat)
    lech = np.abs(toan_bo[hop_le] - da_cat[hop_le])
    hoi_tu = int(np.argmax(lech < 0.01)) if (lech < 0.01).any() else -1
    so_sanh.append(
        {
            "hàm": ten,
            "không ổn định": ten in khong_on_dinh,
            "lệch ở phần tử đầu": round(float(lech[0]), 4),
            "lệch lớn nhất": round(float(lech.max()), 4),
            "phần tử thứ mấy thì lệch < 0,01": hoi_tu,
        }
    )

pd.DataFrame(so_sanh)

,hàm,không ổn định,lệch ở phần tử đầu,lệch lớn nhất,"phần tử thứ mấy thì lệch < 0,01"
0,SMA,False,0.0000,0.0000,0
1,EMA,True,0.4384,0.4384,95
2,RSI,True,9.9407,9.9407,69
3,ADX,True,10.7295,10.7295,115


`SMA` lệch đúng 0 — nó là trung bình của một cửa sổ cố định, không mang trạng
thái. `RSI` và `EMA` mang trạng thái làm trơn theo cấp số nhân, nên giá trị đầu
tiên của chuỗi quyết định một phần kết quả **mãi mãi về sau**, chỉ là tắt dần.

**Hệ quả thực hành:** nếu backtest của bạn dùng `RSI` và bạn đổi `start=` từ
2020 sang 2018, tín hiệu ở năm 2023 **sẽ đổi**. Muốn kết quả tái lập được thì
`start=` phải là một phần của cấu hình backtest, không phải một con số tuỳ hứng.

## 6 · ⚠️ Bẫy 5 — dữ liệu chưa sắp xếp theo thời gian

Ở tầng `finlens.ta.*` không có ai đứng giữa: chuỗi sắp sai cho ra một dãy số
khác hẳn, không lỗi, không cảnh báo.

In [12]:
xao = mot.sample(frac=1, random_state=7)  # xáo trộn thứ tự dòng

ta_xao = finlens.ta.RSI(xao["close"].reset_index(drop=True), timeperiod=14).to_numpy()
ta_dung = finlens.ta.RSI(mot["close"], timeperiod=14).to_numpy()

with warnings.catch_warnings(record=True) as bat:
    warnings.simplefilter("always")
    acc_xao = xao.finlens.rsi(14)

print("Tầng finlens.ta.* trên dữ liệu xáo trộn:")
print(f"  giống kết quả đúng? {np.allclose(np.sort(ta_xao[~np.isnan(ta_xao)]), np.sort(ta_dung[~np.isnan(ta_dung)]))}")
print(f"  giá trị cuối: {ta_xao[-1]:.2f} (xáo) so với {ta_dung[-1]:.2f} (đúng)")

print("\nTầng df.finlens.* trên cùng dữ liệu xáo trộn:")
if bat:
    print(f"  {bat[0].category.__name__}: {str(bat[0].message)[:150]}")
khop = np.allclose(
    acc_xao.sort_values("date")["rsi_14"].to_numpy(),
    mot.finlens.rsi(14)["rsi_14"].to_numpy(),
    equal_nan=True,
)
print(f"  kết quả sau khi sắp lại có khớp không? {khop}")
print(f"  thứ tự dòng trả về có giữ nguyên như đầu vào không? {acc_xao.index.equals(xao.index)}")

Tầng finlens.ta.* trên dữ liệu xáo trộn:
  giống kết quả đúng? False
  giá trị cuối: 51.58 (xáo) so với 50.41 (đúng)

Tầng df.finlens.* trên cùng dữ liệu xáo trộn:
  kết quả sau khi sắp lại có khớp không? True
  thứ tự dòng trả về có giữ nguyên như đầu vào không? True


Tầng `df.finlens.*` tính trên bản đã sắp, trả kết quả về đúng vị trí dòng gốc,
và cảnh báo. Thứ tự dòng bạn nhận về **không đổi** — nên nó không phá code hạ
nguồn của bạn.

## 7 · Dùng thật: một biểu đồ chỉ báo đầy đủ

Ba khung chồng dọc, trục thời gian chung. **Không** chồng RSI lên giá bằng trục
y thứ hai — RSI có thang 0–100 cố định, giá thì không, và ép chúng lên cùng một
khung là để người vẽ chọn hộ người đọc cái gì trông quan trọng.

In [13]:
ma = "HPG"
d = client.eod.stock.ohlcv(ma, start=lui_ngay(HOM_NAY, thang=8)).sort_values("date")
d = d.finlens.sma(20).finlens.sma(50).finlens.rsi(14).finlens.macd()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.52, 0.24, 0.24])

fig.add_trace(go.Candlestick(x=d["date"], open=d["open"], high=d["high"], low=d["low"], close=d["close"],
                             name="Giá",
                             increasing=dict(line=dict(color=TANG, width=1), fillcolor=TANG),
                             decreasing=dict(line=dict(color=GIAM, width=1), fillcolor=GIAM)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=d["date"], y=d["sma_20"], name="SMA 20",
                         line=dict(width=2, color=CHUOI[0])), row=1, col=1)
fig.add_trace(go.Scatter(x=d["date"], y=d["sma_50"], name="SMA 50",
                         line=dict(width=2, color=CHUOI[1])), row=1, col=1)

fig.add_trace(go.Scatter(x=d["date"], y=d["rsi_14"], name="RSI 14",
                         line=dict(width=2, color=CHUOI[6])), row=2, col=1)
fig.add_hline(y=70, line_dash="dot", line_width=1, line_color="#898781", row=2, col=1)
fig.add_hline(y=30, line_dash="dot", line_width=1, line_color="#898781", row=2, col=1)

fig.add_trace(go.Bar(x=d["date"], y=d["macdhist_12_26_9"], name="MACD histogram",
                     marker=dict(color=[TANG if v >= 0 else GIAM for v in d["macdhist_12_26_9"].fillna(0)],
                                 line=dict(width=0))),
              row=3, col=1)
fig.add_trace(go.Scatter(x=d["date"], y=d["macd_12_26_9"], name="MACD",
                         line=dict(width=2, color=CHUOI[0])), row=3, col=1)
fig.add_trace(go.Scatter(x=d["date"], y=d["macdsignal_12_26_9"], name="Signal",
                         line=dict(width=2, color=CHUOI[1])), row=3, col=1)

fig.update_yaxes(title_text="nghìn VND", row=1, col=1)
fig.update_yaxes(title_text="RSI", row=2, col=1, range=[0, 100])
fig.update_yaxes(title_text="MACD", row=3, col=1)
fig.update_layout(
    title_text=f"{ma} — giá, RSI và MACD<br>"
    "<sub style='color:#52514e'>Ba thang khác nhau, ba khung riêng, một trục thời gian chung</sub>",
    height=760,
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
)
fig

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Chỉ báo trên frame nhiều mã | `df.finlens.rsi(14)` — **luôn dùng cái này** |
| Nối nhiều chỉ báo | `df.finlens.rsi(14).finlens.macd().finlens.bbands(20)` |
| Một mảng, một chuỗi | `finlens.ta.RSI(close, timeperiod=14)` |
| Frame chỉ có một chuỗi giá | `df.finlens.rsi(14, by=None)` |
| Biết hàm nào không ổn định | `help(finlens.ta.RSI)` — docstring nói rõ |

**Năm điều mang sang notebook sau:**

1. `talib.RSI` trên frame nhiều mã: mã đầu đúng, **các mã sau sai gần hết**, và
   mọi giá trị sai đều nằm trong khoảng 0–100 hợp lệ. Không có phép kiểm tự
   động nào bắt được.
2. Chỉ báo ra `NaN` ở warm-up, mẫu nến ra `0` — `0` không phân biệt được với
   "không có mẫu".
3. Một `NaN` ở giữa làm hỏng **toàn bộ phần sau**. `ffill` là bịa số, `dropna`
   là đổi cửa sổ; cả hai là quyết định của bạn.
4. 24/75 hàm không ổn định — `start=` phải là một phần cấu hình backtest.
5. Ba thang khác nhau thì ba khung riêng. Không bao giờ hai trục y.

---

**Tiếp theo:** [`32_mau_nen.ipynb`](32_mau_nen.ipynb) — 61 mẫu nến, và vì sao
`df[df.signal == 100]` âm thầm đánh rơi kết quả.